<div style="padding: 20px; background: linear-gradient(90deg, #4b6cb7 0%, #182848 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🔀 Module 6.3: Multi-Query Retriever</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Overcoming bad user prompts by having AI brainstorm alternate queries.</p>
</div>

---

## 1. The "Vocabulary Mismatch" Problem

Semantic search is great, but it can still fail if the user's prompt is completely missing key terminology.
- Document says: "Global warming is primarily caused by human activities."
- User asks: "What causes environmental temperature changes?"

If the vectors don't align close enough, the query misses.

## 2. Multi-Query Solution
Instead of searching once, we pass the user's prompt to an LLM first. The LLM generates **3 to 5 different variations** of the prompt using different terminology. We then search the database for ALL variations and union the results together!

We will build this flow from scratch using Groq to see exactly how it works.

In [1]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
load_dotenv()

docs = [
    Document(page_content="Climate change refers to long-term shifts in temperatures and weather patterns."),
    Document(page_content="Global warming is primarily caused by human activities since the 1800s."),
    Document(page_content="Greenhouse gases trap heat in the atmosphere, raising Earth's temperature."),
]

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="mq_demo")
print("Database loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Database loaded.


In [2]:
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)
    
    # 1. The Prompt that tells the LLM to brainstorm
    prompt = PromptTemplate.from_template(
        """You are an AI language model assistant. Your task is to generate 3 
        different versions of the given user question to retrieve relevant documents from a vector 
        database. By generating multiple perspectives on the user question, your goal is to help 
        the user overcome some of the limitations of the distance-based similarity search. 
        Provide these alternative questions separated by newlines.\n\n"
        Original question: {question}"""
    )
    
    generate_queries_chain = prompt | llm
    
    query = "What causes environmental temperature changes?"
    print(f"USER QUERY: '{query}'\n")
    
    # 2. Generate the variations
    ai_response = generate_queries_chain.invoke({"question": query})
    variations = ai_response.content.strip().split("\n")
    
    print("--- AI BRAINSTORMED VARIATIONS ---")
    for v in variations:
        if v.strip(): print(f"- {v.strip()}")
        
    # 3. Search the Vector Database for ALL queries and union the results
    all_queries = [query] + variations
    unique_docs = set()
    final_docs = []
    
    for q in all_queries:
        if not q.strip(): continue
        docs = vectorstore.similarity_search(q, k=1)
        for doc in docs:
            if doc.page_content not in unique_docs:
                unique_docs.add(doc.page_content)
                final_docs.append(doc)
                
    print("\n--- FINAL UNION RESULTS ---")
    for i, d in enumerate(final_docs):
        print(f"{i+1}. {d.page_content}")
else:
    print("Please provide a GROQ_API_KEY in your .env to see this run.")

USER QUERY: 'What causes environmental temperature changes?'



--- AI BRAINSTORMED VARIATIONS ---
- Here are three alternative versions of the user question to help overcome some of the limitations of distance-based similarity search:
- Changes in environmental temperature: What are the primary factors influencing this phenomenon?
- To gain a better understanding of temperature fluctuations: Identify the underlying mechanisms driving these changes.
- What are the key drivers of variability in global environmental temperature patterns, and how do they interact with one another?

--- FINAL UNION RESULTS ---
1. Greenhouse gases trap heat in the atmosphere, raising Earth's temperature.
2. Global warming is primarily caused by human activities since the 1800s.
3. Climate change refers to long-term shifts in temperatures and weather patterns.
